# TARDIS — Analytics des retards ferroviaires (EDA & cleaning)

## Fil conducteur

1. **Problème** — Jeu tabulaire OD × mois pour analyser punctuality, retards, annulations.
2. **Qualité** — Formats mixtes, NA, labels bruités, doublons, colonnes commentaires (hors scope NLP).
3. **Cleaning** — Preuve visuelle (avant/après), puis normalisation stations / `Service`, types, doublons.
4. **Features** — Calendrier, `route`, ratios métier (`cancel_rate`, `severe_delay_rate`, …).
5. **EDA** — Distributions, corrélations, série temporelle, gares / routes.
6. **Livrable** — `cleaned_dataset.csv`.

## 1. Imports & configuration

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

FIG = {"sm": (10, 5), "md": (10, 6), "lg": (12, 5), "xl": (14, 7), "sq": (8, 6), "heat": (14, 10)}
PAL = {"delay": "#4a90a4", "route": "crest", "box": "pastel"}
PAL_CANCEL_BAR = "rocket"
CMAP = {"missing": "YlOrRd", "corr": "coolwarm"}

sns.set_theme(style="whitegrid")
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.titlesize"] = 14

## 2. Chargement

### 2.0 Lecture CSV + `strip` des noms de colonnes

Juste après ces deux lignes, on crée **`raw_df = df.copy()`** : c’est la **référence « avant nettoyage »** (plus tard le notebook modifie **`df`** seulement ; **`raw_df`** ne doit pas être réécrit).

In [ ]:
CSV_PATH = "dataset.csv"

df = pd.read_csv(CSV_PATH, sep=";", na_values=["N/A", "", " "], engine="python")
df.columns = df.columns.str.strip()

# Snapshot avant tout nettoyage : obligatoire pour les sections 6 et 7 (avant / après)
raw_df = df.copy()

print("Shape df (après strip colonnes) :", df.shape)
print("Exemple de colonnes :", list(df.columns[:6]), "...")
print("raw_df cree : copie independante de df ?", raw_df is not df)

### 2.1 La variable `raw_df` (preuve dans les sorties)

La cellule suivante affiche explicitement **`raw_df`** (aperçu). Si tu ne vois pas **`raw_df`** dans ton IDE, recharge le fichier depuis le disque (**Revert File**) : tu regardes peut‑être une ancienne version du notebook.

In [ ]:
# Reference avant nettoyage — meme contenu que df a ce stade, objet pandas distinct
assert raw_df is not df
print("raw_df.shape :", raw_df.shape)
print("Colonnes (extrait) :", list(raw_df.columns[:8]))
display(raw_df.head(3))

## 3. Inspection initiale

In [ ]:
display(df.head())

print("\n--- info() ---")
df.info()

print("\n--- describe() — numériques ---")
display(df.describe())

print("\n--- describe(include=['object']) ---")
display(df.describe(include=["object"]))

## 4. Valeurs manquantes (audit)

In [ ]:
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct}).sort_values(
    "missing_pct", ascending=False
)
display(missing_df)

sample_size = min(800, len(df))
rng = np.random.default_rng(42)
sample_idx = rng.choice(df.index, size=sample_size, replace=False)
missing_sample = df.loc[sample_idx].isna()

fig, ax = plt.subplots(figsize=FIG["xl"])
sns.heatmap(missing_sample.T, cbar=True, yticklabels=True, cmap=CMAP["missing"], ax=ax)
ax.set_title("Valeurs manquantes — échantillon aleatoire", fontsize=13)
ax.set_xlabel("Indice des lignes (echantillon)")
ax.set_ylabel("Colonne")
plt.tight_layout()
plt.show()

tot_cells = missing_sample.shape[0] * missing_sample.shape[1]
share_miss = missing_sample.to_numpy().mean() * 100
print(f"Taux global de cellules manquantes dans l'echantillon : {share_miss:.2f} % (sur {tot_cells:,} cellules).")

## 5. Doublons (avant nettoyage métier)

In [ ]:
n_dup = df.duplicated().sum()
pct_dup = 100 * n_dup / len(df)
print(f"Lignes dupliquees : {n_dup} ({pct_dup:.3f} % du fichier brut).")

## 6. Preuve visuelle du nettoyage (état brut)

Avant d’appliquer les transformations définitives, on documente :

1. **Commentaires** — échantillon du bruit textuel (colonnes supprimées ensuite).
2. **Dates invalides** — pas seulement le décompte : exemples de chaînes qui ne parsèment pas en `%Y-%m`.
3. **Colonnes `object` à forte teneur numérique** — quelles colonnes sont encore typées object alors qu’elles devraient être numériques ; **valeurs problématiques** (`"-"`, `unknown`, séparateur décimal `,`, etc.).
4. **`Service`** — cardinalité brute et extrait des modalités les plus fréquentes.
5. **Stations** — cardinalité départ / arrivée avant normalisation texte.

*(Les métriques « après » viennent après la section nettoyage.)*

### 6.1 Commentaires : bruit réel avant suppression

In [ ]:
COMMENT_PREVIEW = [
    "Cancellation comments",
    "Departure delay comments",
    "Arrival delay comments",
]
for col in COMMENT_PREVIEW:
    if col not in raw_df.columns:
        continue
    s = raw_df[col].dropna()
    if len(s) == 0:
        print(f"{col}: (vide)")
        continue
    print(f"\n--- Echantillon aleatoire : {col} ---")
    display(s.sample(min(5, len(s)), random_state=42))

### 6.2 Dates : valeurs invalides (pas seulement le total)

In [ ]:
if "Date" in raw_df.columns:
    parsed = pd.to_datetime(raw_df["Date"], format="%Y-%m", errors="coerce")
    invalid_mask = parsed.isna() & raw_df["Date"].notna()
    n_invalid = int(invalid_mask.sum())
    print(f"Lignes avec Date non parsable (non NA brute mais NaT apres coerce) : {n_invalid}")
    if n_invalid > 0:
        bad = raw_df.loc[invalid_mask, "Date"]
        print("Exemples de valeurs brutes problematiques (uniques, max 15) :")
        print(bad.astype(str).unique()[:15])
        print("\nApercu des premieres lignes concernees :")
        display(raw_df.loc[invalid_mask].head(10))
else:
    print("Colonne Date absente.")

### 6.3 Colonnes `object` → numériques : diagnostic et valeurs problématiques

In [ ]:
object_cols = list(raw_df.select_dtypes(include=["object"]).columns)
numeric_like = []
for col in object_cols:
    conv = pd.to_numeric(raw_df[col], errors="coerce")
    nn = raw_df[col].notna().sum()
    if nn == 0:
        continue
    ok = conv.notna().sum()
    if ok / nn >= 0.5:
        numeric_like.append(col)

print("Colonnes encore 'object' mais majoritairement convertibles en numerique :")
print(numeric_like if numeric_like else "(aucune detectee avec seuil 50 %)")

for col in numeric_like:
    problematic = raw_df.loc[
        pd.to_numeric(raw_df[col], errors="coerce").isna() & raw_df[col].notna(), col
    ]
    if len(problematic) > 0:
        print(f"\nValeurs problematiques dans `{col}` (echantillon unique, max 10) :")
        print(problematic.astype(str).unique()[:10])

### 6.4 `Service` et stations : cardinalité brute

In [ ]:
if "Service" in raw_df.columns:
    print(f"Service — nombre de modalites distinctes (brut) : {raw_df['Service'].nunique()}")
    print("\nTop modalites (brut) :")
    display(raw_df["Service"].astype(str).value_counts().head(15))

for col in ["Departure station", "Arrival station"]:
    if col in raw_df.columns:
        print(f"\n{col} — cardinalite brute : {raw_df[col].nunique()}")

## 7. Nettoyage appliqué

Étapes : suppression des colonnes commentaires, normalisation texte des stations, canonicalisation fuzzy **`Service`** → **`Service_clean`** (+ encodage optionnel), **`Date`** au format mensuel, conversion numérique là où c’est cohérent, puis suppression des doublons.

In [ ]:
import difflib

COMMENT_DROP = [
    "Cancellation comments",
    "Departure delay comments",
    "Arrival delay comments",
]
present = [c for c in COMMENT_DROP if c in df.columns]
if present:
    df = df.drop(columns=present)
    print("Colonnes commentaires supprimees :", present)

for col in ["Departure station", "Arrival station"]:
    if col in df.columns:
        df[col] = (
            df[col].astype("string")
            .str.lower()
            .str.strip()
            .str.replace(r"[-]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
        )

manual = {"paris gare de lyon": "paris", "paris-gare-de-lyon": "paris"}
for col in ["Departure station", "Arrival station"]:
    if col in df.columns:
        df[col] = df[col].replace(manual)

if "Service" in df.columns:
    svc_alpha = df["Service"].astype("string").str.strip().str.lower().str.replace(r"[^a-z]", "", regex=True)
    TARGETS = ["national", "international"]

    def canon(v):
        if pd.isna(v) or v == "":
            return pd.NA
        if "inter" in v or "intern" in v:
            return "International"
        if "natio" in v or "naton" in v:
            return "National"
        m = difflib.get_close_matches(v, TARGETS, n=1, cutoff=0.55)
        return m[0].title() if m else pd.NA

    df["Service_clean"] = svc_alpha.apply(canon)
    obs = df["Service_clean"].dropna().unique().tolist()
    if len(obs) == 2:
        le = LabelEncoder()
        m = df["Service_clean"].notna()
        enc = pd.Series(np.nan, index=df.index, dtype="float64")
        enc.loc[m] = le.fit_transform(df.loc[m, "Service_clean"]).astype(float)
        df["Service_encoded"] = enc
        print("Encodage Service :", dict(zip(le.classes_, le.transform(le.classes_))))

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m", errors="coerce")
print("Dates invalides (NaT) :", int(df["Date"].isna().sum()))

for col in df.columns:
    if df[col].dtype == "object":
        conv = pd.to_numeric(df[col], errors="coerce")
        nn, ok = df[col].notna().sum(), conv.notna().sum()
        if nn > 0 and ok / nn >= 0.5:
            df[col] = conv

df = df.drop_duplicates(keep="first").reset_index(drop=True)

display(df.dtypes.to_frame("dtype"))

### 7.1 Après nettoyage : comparaisons chiffrées & summary

In [ ]:
rows = []

if "Service" in raw_df.columns and "Service_clean" in df.columns:
    before_svc = raw_df["Service"].nunique()
    after_svc = df["Service_clean"].nunique()
    rows.append({"metric": "Service — modalites distinctes", "before": before_svc, "after": after_svc})
    print(f"Unique Service (brut) → Service_clean : {before_svc} → {after_svc}")

for col in ["Departure station", "Arrival station"]:
    if col in raw_df.columns and col in df.columns:
        b = raw_df[col].nunique()
        a = df[col].nunique()
        rows.append({"metric": f"{col} — cardinalite", "before": b, "after": a})
        print(f"{col} — cardinalite : {b} → {a} (reduction bruit / fusion de variantes)")

converted = []
for col in df.columns:
    if col not in raw_df.columns:
        continue
    if raw_df[col].dtype == object and pd.api.types.is_numeric_dtype(df[col]):
        converted.append(col)
print("\nColonnes passees de object → numerique apres cleaning :")
print(converted if converted else "(aucune ou deja numeriques en amont)")

summary_df = pd.DataFrame(rows)
display(summary_df)

print("\n--- Distribution Service_clean (apres) ---")
if "Service_clean" in df.columns:
    display(df["Service_clean"].value_counts(dropna=False))

### 7.2 Tableau récapitulatif du cleaning

In [ ]:
_comment_drop_names = [
    "Cancellation comments",
    "Departure delay comments",
    "Arrival delay comments",
]
cleaning_summary = pd.DataFrame(
    {
        "Etape": [
            "Colonnes commentaires supprimees",
            "Stations normalisees (lower, tirets, espaces)",
            "Service → Service_clean (fuzzy National / International)",
            "Date parsee %Y-%m (NaT si invalide)",
            "Coercion numerique sur colonnes object majoritairement numeriques",
            "Doublons supprimes",
        ],
        "Detail": [
            str([c for c in _comment_drop_names if c in raw_df.columns]),
            "Departure / Arrival station",
            "difflib + regles inter/natio",
            "pd.to_datetime(..., errors='coerce')",
            "pd.to_numeric(..., errors='coerce') si >= 50 % convertibles",
            "drop_duplicates(keep='first')",
        ],
    }
)
display(cleaning_summary)
print("Shape brut → shape nettoye :", raw_df.shape, "→", df.shape)

## 8. Feature engineering — ratios & temporalite

In [ ]:
SCHED = "Number of scheduled trains"
CANC = "Number of cancelled trains"
DEP_DEL = "Number of trains delayed at departure"
SEV = "Number of trains delayed > 60min"
DEL_ARR = "Average delay of all trains at arrival"

df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

if "Departure station" in df.columns and "Arrival station" in df.columns:
    df["route"] = (
        df["Departure station"].fillna("unknown_dep").astype(str)
        + "_"
        + df["Arrival station"].fillna("unknown_arr").astype(str)
    )

if SCHED in df.columns:
    if CANC in df.columns:
        df["cancel_rate"] = np.where(df[SCHED] > 0, df[CANC] / df[SCHED], np.nan)
    if DEP_DEL in df.columns:
        df["delay_rate_departure"] = np.where(df[SCHED] > 0, df[DEP_DEL] / df[SCHED], np.nan)
    if SEV in df.columns:
        df["severe_delay_rate"] = np.where(df[SCHED] > 0, df[SEV] / df[SCHED], np.nan)

display(
    df[
        [
            "year",
            "month",
            "quarter",
            "route",
            "cancel_rate",
            "delay_rate_departure",
            "severe_delay_rate",
            "month_sin",
            "month_cos",
        ]
    ].head(10)
)

## 9. Visualisations exploratoires

### 9.1 Distribution du retard moyen a l'arrivee

In [ ]:
DEL_ARR = "Average delay of all trains at arrival"

if DEL_ARR in df.columns:
    d = df[DEL_ARR].dropna()
    q25, q50, q75, q90 = d.quantile([0.25, 0.5, 0.75, 0.9])
    pct_le_q75 = (d <= q75).mean() * 100
    pct_ge_q90 = (d >= q90).mean() * 100
    sk = d.skew()

    fig, ax = plt.subplots(figsize=FIG["sm"])
    sns.histplot(d, kde=True, bins=40, color=PAL["delay"], ax=ax)
    ax.axvline(q75, color="darkred", linestyle="--", linewidth=1, label=f"Q3 = {q75:.2f} min")
    ax.axvline(q90, color="orange", linestyle=":", linewidth=1, label=f"P90 = {q90:.2f} min")
    ax.set_title("Distribution — retard moyen a l'arrivee (OD x mois)")
    ax.set_xlabel("Minutes")
    ax.set_ylabel("Frequence")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("--- Synthese quantitative ---")
    print(f" n valide = {len(d)}")
    print(f" Skewness (Pearson) = {sk:.3f}")
    print(f" Q1 = {q25:.2f} min | mediane = {q50:.2f} min | Q3 = {q75:.2f} min | P90 = {q90:.2f} min")
    print(
        f" Environ {pct_le_q75:.1f} % des observations sont au plus au Q3 ({q75:.2f} min),"
        f" tandis que ~{pct_ge_q90:.1f} % depassent le 90e percentile."
    )

### 9.2 Boxplot — Service_clean

In [ ]:
if "Service_clean" in df.columns and DEL_ARR in df.columns:
    plot_df = df.dropna(subset=["Service_clean", DEL_ARR])
    fig, ax = plt.subplots(figsize=FIG["sq"])
    sns.boxplot(data=plot_df, x="Service_clean", y=DEL_ARR, palette=PAL["box"], ax=ax)
    ax.set_title("Retard moyen a l'arrivee — National vs International")
    ax.set_xlabel("Service (nettoye)")
    ax.set_ylabel("Minutes")
    plt.tight_layout()
    plt.show()

    for svc in plot_df["Service_clean"].dropna().unique():
        sub = plot_df.loc[plot_df["Service_clean"] == svc, DEL_ARR]
        print(f" {svc}: mediane = {sub.median():.2f} min | n = {len(sub)}")

### 9.3 Top 15 gares de depart

In [ ]:
ST = "Departure station"
if DEL_ARR in df.columns and ST in df.columns:
    top = df.groupby(ST, observed=True)[DEL_ARR].mean().sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=FIG["md"])
    sns.barplot(x=top.values, y=top.index, hue=top.index, palette=PAL["route"], legend=False, ax=ax)
    ax.set_title("Top 15 gares de depart — retard moyen a l'arrivee")
    ax.set_xlabel("Retard moyen (minutes)")
    plt.tight_layout()
    plt.show()

    worst = top.index[0]
    print(f" Gare #1 : {worst} — {top.iloc[0]:.2f} min en moyenne.")

### 9.4 Routes & hotspots

In [ ]:
MIN_N = 3
if "route" in df.columns and DEL_ARR in df.columns:
    g = df.groupby("route", observed=True)[DEL_ARR].agg(["mean", "count"])
    g = g[g["count"] >= MIN_N].sort_values("mean", ascending=False)
    worst_routes_delay = g.head(10).reset_index()
    worst_routes_delay.columns = ["route", "mean_delay_arr", "n_obs"]
    print("--- Top 10 routes — retard moyen (n >= %d) ---" % MIN_N)
    display(worst_routes_delay)

if "route" in df.columns and "cancel_rate" in df.columns:
    gc = df.groupby("route", observed=True)["cancel_rate"].agg(["mean", "count"])
    gc = gc[gc["count"] >= MIN_N].sort_values("mean", ascending=False)
    hotspots_cancel = gc.head(10).reset_index()
    hotspots_cancel.columns = ["route", "mean_cancel_rate", "n_obs"]
    print("--- Top 10 routes — cancel_rate moyen (n >= %d) ---" % MIN_N)
    display(hotspots_cancel)

    fig, ax = plt.subplots(figsize=FIG["lg"])
    plot_r = gc.head(10).sort_values("mean").reset_index()
    sns.barplot(data=plot_r, x="mean", y="route", palette=PAL_CANCEL_BAR, ax=ax)
    ax.set_title("Top 10 routes — taux d'annulation moyen (hotspots)")
    ax.set_xlabel("cancel_rate moyen")
    plt.tight_layout()
    plt.show()

### 9.5 Serie temporelle mensuelle

In [ ]:
if DEL_ARR in df.columns and "Date" in df.columns:
    ts = df.dropna(subset=["Date", DEL_ARR])
    monthly_delay = ts.groupby("Date", observed=True)[DEL_ARR].mean()

    fig, ax = plt.subplots(figsize=FIG["lg"])
    sns.lineplot(x=monthly_delay.index, y=monthly_delay.values, marker="o", color=PAL["delay"], ax=ax)
    ax.set_title("Retard moyen a l'arrivee — evolution mensuelle (reseau)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Minutes")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    cv = monthly_delay.std() / monthly_delay.mean() if monthly_delay.mean() != 0 else np.nan
    print(f" Coefficient de variation mensuel (std/mean) : {cv:.3f}")

### 9.6 Matrice de correlations

In [ ]:
DEL_ARR = "Average delay of all trains at arrival"
clean_df = df

corr = clean_df.select_dtypes(include=np.number).corr().round(2)

fig, ax = plt.subplots(figsize=FIG["heat"])
sns.heatmap(corr, cmap=CMAP["corr"], center=0, linewidths=0.3, annot=False, ax=ax)
ax.set_title("Correlation matrix (corr arrondie a 2 decimales)")
plt.tight_layout()
plt.show()

pairs = [
    ("Number of cancelled trains", DEL_ARR),
    ("Average delay of all trains at departure", "Average delay of all trains at arrival"),
    ("Number of cancelled trains", "Number of trains delayed > 60min"),
]
pairs = [(a, b) for a, b in pairs if a in clean_df.columns and b in clean_df.columns]
print("Correlations de Pearson (paires cles) :")
for a, b in pairs:
    c0 = clean_df[a].corr(clean_df[b])
    print(f"  {c0:.4f}  |  {a[:45]}  vs  {b[:45]}")

### 9.7 Scatter annulations vs retard

In [ ]:
XC = "Number of cancelled trains"
if XC in df.columns and DEL_ARR in df.columns:
    r_sc = df[XC].corr(df[DEL_ARR])
    fig, ax = plt.subplots(figsize=FIG["sq"])
    sns.scatterplot(data=df, x=XC, y=DEL_ARR, alpha=0.35, s=20, color=PAL["delay"], ax=ax)
    ax.set_title("Annulations vs retard moyen a l'arrivee")
    ax.set_xlabel("Nombre de trains annules")
    ax.set_ylabel("Retard moyen arrivee (minutes)")
    plt.tight_layout()
    plt.show()
    print(f" Pearson (annulations vs retard moyen arrivee) : {r_sc:.4f}")

## 10. Export

In [ ]:
OUT_PATH = "cleaned_dataset.csv"
df.to_csv(OUT_PATH, index=False)
print("Fichier ecrit :", OUT_PATH)
print("Shape final :", df.shape)

## 11. Limitations

- Colonnes commentaires retirees (pas d'analyse NLP des causes).
- Normalisation des gares rule-based ; variantes possibles.
- Granularite mois x OD — pas d'analyse horaire fine.
- Correlations lineaires seules.

## 12. Synthese — insights metier

Asymetrie des retards, hotspots gares/routes, serie temporelle pour monitoring / forecasting, vision multi-KPI.

## 13. Key Findings

In [ ]:
DEL_ARR = "Average delay of all trains at arrival"
d = df[DEL_ARR].dropna()
sk = float(d.skew())
q75 = float(d.quantile(0.75))
pct_le_q75 = float((d <= q75).mean() * 100)
r_cd = df["Number of cancelled trains"].corr(df[DEL_ARR]) if "Number of cancelled trains" in df.columns else np.nan
r_da = (
    df["Average delay of all trains at departure"].corr(df[DEL_ARR])
    if "Average delay of all trains at departure" in df.columns
    else np.nan
)

print("Key Findings (this dataset, OD-month grain)")
print("-" * 55)
print(f"• Delay distribution: skewness ~ {sk:.2f} — tail behaviour matters for KPIs.")
print(f"• ~{pct_le_q75:.0f}% of observations fall at or below Q3 (~{q75:.1f} min).")
if "route" in df.columns:
    print("• Route-level rankings (see sec. 9.4): prioritize OD pairs with high delay or cancel_rate.")
print(f"• Pearson: cancellations vs arrival delay ~ {r_cd:.3f}; dep vs arr delay ~ {r_da:.3f}.")
print("• Monthly aggregation: use time-based validation for modelling.")
print("-" * 55)

- **Delay distributions** are often **asymmetric**; Q3 / P90 separate routine OD-months from severe episodes.
- **Routes** can rank high on mean delay or **cancel_rate** — check printed tables.
- **Cancellation vs delay** shows a **modest linear association** at this grain; combine with **rates** and routes.

*Validate on fresh data before executive decisions.*